# Regulatory Corpus: Deep Cleaning, Structuring & Semantic Analysis

**Research:** API-Mediated AI Safety Asymmetry in Indonesia  
**Purpose:** Clean, structure, and semantically interrogate Indonesian AI regulatory documents  
**Documents:**
- Strategi Nasional Kecerdasan Artifisial 2020-2045 (*Stranas KA*)
- UU Nomor 1 Tahun 2024 (*UU ITE Amandemen*)
- UU Nomor 27 Tahun 2022 (*UU PDP*)

**Analytical Framework:** Based on *reenhance-concept-v2.md* — Phase 4: Deep Regulatory Analysis

---

## Pipeline Overview

```
Raw OCR Text
     │
     ▼
Comprehensive Cleaning  ──→  docs/regulatory_corpus/cleaned/
     │
     ▼
Structural Parsing  ──→  BAB / Pasal / Ayat hierarchy JSON
     │
     ▼
Semantic Coverage Analysis  ──→  AI Safety Concept Heatmap
     │
     ▼
Liability Actor Mapping  ──→  Who bears responsibility?
     │
     ▼
Regulatory Gap Matrix  ──→  data/processed/regulatory_gap_analysis.json
```

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Uncomment on first run or when running in Colab
# !pip install -q sentence-transformers scikit-learn matplotlib seaborn pandas
# !pip install -q transformers torch  # Only for LLM-based analysis

In [ ]:
import os
import re
import json
import unicodedata
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime

print("Core imports OK.")

# Sentence-transformers for semantic analysis (CPU-compatible)
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    SEMANTIC_AVAILABLE = True
    print("sentence-transformers available — semantic analysis enabled.")
except ImportError:
    SEMANTIC_AVAILABLE = False
    print("WARNING: sentence-transformers not installed. Run: pip install sentence-transformers")
    print("Keyword-based fallback will be used for coverage analysis.")

In [ ]:
# ── Environment Configuration ─────────────────────────────────────────────────
# Detect Colab vs. local execution
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Colab Notebooks/is_de')  # ← adjust to your Drive path
    print("Running in Google Colab — Drive mounted.")
except ImportError:
    IN_COLAB = False
    # Resolve relative to this notebook's location
    BASE_PATH = Path(globals().get('__vsc_ipynb_file__', __file__)).resolve().parent.parent \
        if '__vsc_ipynb_file__' in globals() \
        else Path(r'd:\BINUS Works\Codes\research_banks\research\is_de')
    print(f"Running locally — base path: {BASE_PATH}")

# Corpus paths
RAW_CORPUS_DIR  = BASE_PATH / 'docs' / 'regulatory_corpus'
CLEAN_CORPUS_DIR = BASE_PATH / 'docs' / 'regulatory_corpus' / 'cleaned'
PROCESSED_DIR   = BASE_PATH / 'data' / 'processed'
DIAGRAMS_DIR    = BASE_PATH / 'diagrams'

CLEAN_CORPUS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DIAGRAMS_DIR.mkdir(parents=True, exist_ok=True)

# Document registry
DOCUMENTS = {
    'stranas_ka':  {'file': 'stranas-ka-2045.txt',      'short': 'Stranas KA',  'type': 'policy_strategy'},
    'uu_ite_2024': {'file': 'UU Nomor 1 Tahun 2024.txt','short': 'UU ITE 2024', 'type': 'statute'},
    'uu_pdp_2022': {'file': 'UU Nomor 27 Tahun 2022.txt','short': 'UU PDP 2022','type': 'statute'},
}

print(f"\nCorpus directory: {RAW_CORPUS_DIR}")
for doc_id, meta in DOCUMENTS.items():
    p = RAW_CORPUS_DIR / meta['file']
    status = '✓' if p.exists() else '✗ MISSING'
    print(f"  {status}  {meta['short']} ({meta['file']})")

In [ ]:
# ── Load Raw Corpus ───────────────────────────────────────────────────────────
raw_texts = {}
for doc_id, meta in DOCUMENTS.items():
    path = RAW_CORPUS_DIR / meta['file']
    try:
        text = path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        text = path.read_text(encoding='latin-1')
    raw_texts[doc_id] = text
    print(f"{meta['short']:15s}  {len(text):>8,} chars  {len(text.split()):>8,} words")

In [ ]:
# ── Comprehensive OCR Cleaning Functions ─────────────────────────────────────

# Expanded OCR correction dictionary sourced from observed artefacts
OCR_CORRECTIONS = {
    # Garbled header terms
    'KECERDASAN INDONESIA': 'KECERDASAN ARTIFISIAL',
    r'\bINDONESIA\b(?=\s+ARTIFISIAL|\s+KA\b)': 'ARTIFISIAL',
    # Common OCR character substitutions in Indonesian legal text
    r'\bNEPLTBLIK\b': 'REPUBLIK',
    r'\bNEPUELIK\b':  'REPUBLIK',
    r'\bNEPUBUK\b':   'REPUBLIK',
    r'\bREPIIBUK\b':  'REPUBLIK',
    r'\bREPIJBUK\b':  'REPUBLIK',
    r'\bREPIITEUK\b': 'REPUBLIK',
    r'\bPRES!DEN\b':  'PRESIDEN',
    r'\bINDONE!3IA\b':'INDONESIA',
    r'\bINDONESI\.A\b':'INDONESIA',
    r'\bUNDANG\.UNDANG\b': 'UNDANG-UNDANG',
    r'\bivlenimbang\b': 'Menimbang',
    r'\btr\{I\b': 'dan',
    r'\btr\{ir\b': 'dan',
    r'\btr\{rr\b': 'dan',
    r'\bT\{IItrtrIXNItrIIFEIn\b': '',
    # Fix split words with dots in statute text
    r'(?<=\w)\.(?=\w{2,})': '-',   # e.g., Pasal.5 → Pasal-5 (only if next is 2+ chars)
    # Fix Rp formatting
    r'Rp\s+': 'Rp',
}

REMOVE_PATTERNS = [
    r'SK\s+No\s+\d+\s*[A-Z]?',       # Page markers: SK No XXXXXX A
    r'^\s*\d+\s*\|\s*',               # Line numbers: 1 | 
    r'\bSALINAN\b',                    # Header watermark
    r'T\{IItrtrIX[\w]*',               # Garbled OCR block
    r'Lembaran\s+Negara[^\n]*',        # Footer references
    r'Tambahan\s+Lembaran\s+Negara[^\n]*',
    r'-\s*\d+\s*-',                    # Page numbers like -2-
    r'www\.[^\s]+',                    # URLs in headers
]


def normalize_whitespace(text: str) -> str:
    """Collapse multiple spaces/tabs; preserve single newlines between paragraphs."""
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def fix_broken_sentences(text: str) -> str:
    """Rejoin lines that were broken mid-sentence (no terminal punctuation)."""
    lines = text.split('\n')
    merged = []
    buffer = ''
    for line in lines:
        line = line.strip()
        if not line:
            if buffer:
                merged.append(buffer)
                buffer = ''
            merged.append('')
            continue
        # If the line is a section header (ALL CAPS or starts with 'Pasal'/'BAB'), keep separate
        if re.match(r'^(BAB|Pasal|Ayat|PASAL|BAGIAN|Paragraf)\b', line) or line.isupper():
            if buffer:
                merged.append(buffer)
                buffer = ''
            merged.append(line)
        elif buffer and not buffer.rstrip().endswith(('.', ';', ':')):
            buffer += ' ' + line
        else:
            if buffer:
                merged.append(buffer)
            buffer = line
    if buffer:
        merged.append(buffer)
    return '\n'.join(merged)


def clean_document(text: str) -> str:
    """Apply the full cleaning pipeline to a single regulatory document."""
    # 1. Unicode normalization
    text = unicodedata.normalize('NFKC', text)

    # 2. Remove boilerplate patterns
    for pattern in REMOVE_PATTERNS:
        text = re.sub(pattern, '', text, flags=re.MULTILINE | re.IGNORECASE)

    # 3. Apply OCR corrections
    for pattern, replacement in OCR_CORRECTIONS.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    # 4. Fix broken sentences
    text = fix_broken_sentences(text)

    # 5. Final whitespace normalization
    text = normalize_whitespace(text)

    return text


print("Cleaning functions defined.")

In [ ]:
# ── Apply Cleaning Pipeline ───────────────────────────────────────────────────
cleaned_texts = {}
cleaning_stats = {}

for doc_id, raw in raw_texts.items():
    cleaned = clean_document(raw)
    cleaned_texts[doc_id] = cleaned

    raw_words   = len(raw.split())
    clean_words = len(cleaned.split())
    raw_chars   = len(raw)
    clean_chars = len(cleaned)

    cleaning_stats[doc_id] = {
        'raw_chars': raw_chars,
        'clean_chars': clean_chars,
        'raw_words': raw_words,
        'clean_words': clean_words,
        'chars_removed': raw_chars - clean_chars,
        'reduction_pct': round((raw_chars - clean_chars) / raw_chars * 100, 1),
    }

    short = DOCUMENTS[doc_id]['short']
    print(f"[{short}]")
    print(f"  Before: {raw_chars:,} chars / {raw_words:,} words")
    print(f"  After:  {clean_chars:,} chars / {clean_words:,} words")
    print(f"  Removed: {cleaning_stats[doc_id]['chars_removed']:,} chars ({cleaning_stats[doc_id]['reduction_pct']}%)")
    print()

In [ ]:
# ── Structural Parser: BAB / Pasal / Ayat Hierarchy ───────────────────────────

def parse_legal_structure(text: str, doc_type: str = 'statute') -> list:
    """
    Parse Indonesian legal text into a hierarchy of sections.
    Returns a list of dicts: {level, number, title, content}
    """
    sections = []

    # Patterns for Indonesian legal document structure
    bab_pattern    = re.compile(r'^BAB\s+((?:I{1,3}|IV|VI{0,3}|IX|X{0,3})(?:I{1,3}|V)?|\d+)\s*(.*)$', re.MULTILINE | re.IGNORECASE)
    pasal_pattern  = re.compile(r'^Pasal\s+(\d+(?:[a-z])?)\s*$', re.MULTILINE | re.IGNORECASE)
    bagian_pattern = re.compile(r'^BAGIAN\s+(\w+)\s*(.*)$', re.MULTILINE | re.IGNORECASE)

    lines = text.split('\n')
    current_bab    = None
    current_bagian = None
    current_pasal  = None
    current_buffer = []

    def flush_pasal():
        if current_pasal and current_buffer:
            sections.append({
                'level':   'pasal',
                'bab':     current_bab,
                'bagian':  current_bagian,
                'number':  current_pasal,
                'content': ' '.join(current_buffer).strip(),
                'word_count': len(' '.join(current_buffer).split()),
            })

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue

        bab_m    = bab_pattern.match(stripped)
        pasal_m  = pasal_pattern.match(stripped)
        bagian_m = bagian_pattern.match(stripped)

        if bab_m:
            flush_pasal()
            current_buffer = []
            current_pasal  = None
            current_bab    = f"BAB {bab_m.group(1)} {bab_m.group(2)}".strip()
            current_bagian = None
        elif bagian_m:
            flush_pasal()
            current_buffer = []
            current_pasal  = None
            current_bagian = f"BAGIAN {bagian_m.group(1)} {bagian_m.group(2)}".strip()
        elif pasal_m:
            flush_pasal()
            current_buffer = []
            current_pasal = pasal_m.group(1)
        else:
            current_buffer.append(stripped)

    flush_pasal()  # flush the last pasal
    return sections


# Parse all documents
structured_corpus = {}
for doc_id, text in cleaned_texts.items():
    doc_type = DOCUMENTS[doc_id]['type']
    sections = parse_legal_structure(text, doc_type)
    structured_corpus[doc_id] = sections
    short = DOCUMENTS[doc_id]['short']
    pasals = [s for s in sections if s['level'] == 'pasal']
    print(f"{short}: {len(sections)} sections parsed ({len(pasals)} 'Pasal' nodes)")

In [ ]:
# ── Export Cleaned Corpus ─────────────────────────────────────────────────────
for doc_id, text in cleaned_texts.items():
    out_path = CLEAN_CORPUS_DIR / DOCUMENTS[doc_id]['file']
    out_path.write_text(text, encoding='utf-8')
    print(f"Saved: {out_path.name}  ({len(text):,} chars)")

# Export structured JSON (BAB/Pasal hierarchy)
structured_export = {}
for doc_id, sections in structured_corpus.items():
    structured_export[doc_id] = {
        'metadata': DOCUMENTS[doc_id],
        'sections': sections,
        'total_sections': len(sections),
        'total_pasals': len([s for s in sections if s['level'] == 'pasal']),
        'total_words':  sum(s.get('word_count', 0) for s in sections),
    }

structured_path = PROCESSED_DIR / 'regulatory_structured.json'
structured_path.write_text(
    json.dumps(structured_export, indent=2, ensure_ascii=False),
    encoding='utf-8'
)
print(f"\nStructured corpus exported: {structured_path}")

# Cleaning report
report_lines = [f"# Regulatory Corpus Cleaning Report\n",
                f"Generated: {datetime.now().isoformat()}\n\n"]
for doc_id, stats in cleaning_stats.items():
    short = DOCUMENTS[doc_id]['short']
    report_lines.append(f"## {short}\n")
    for k, v in stats.items():
        report_lines.append(f"- {k}: {v}\n")
    report_lines.append("\n")

report_path = CLEAN_CORPUS_DIR / 'cleaning_report_v2.md'
report_path.write_text(''.join(report_lines), encoding='utf-8')
print(f"Cleaning report: {report_path}")

## Phase 4: Semantic Safety Coverage Analysis

We use multilingual sentence embeddings (`paraphrase-multilingual-MiniLM-L12-v2`) to compute semantic similarity between each regulatory document and a set of AI safety concepts derived from the research framework.

**Key Research Question:** Do Indonesian AI regulations address the specific risks of API-mediated deployments, or does a systematic coverage gap exist?

In [ ]:
# ── AI Safety Concept Definitions ─────────────────────────────────────────────

SAFETY_CONCEPTS = {
    # ---------- API & Deployment-Specific ----------------------------------------
    'API Safety Obligation':       'kewajiban keamanan antarmuka pemrograman aplikasi AI',
    'API Developer Liability':     'tanggung jawab hukum pengembang yang menggunakan API kecerdasan buatan',
    'Foundation Model Provider':   'penyedia model dasar kecerdasan artifisial dari luar negeri',
    'Third-party Deployment':      'penerapan AI oleh pihak ketiga melalui antarmuka API',
    'Safety Guardrail Stripping':  'penghapusan sistem pengaman keamanan pada model AI',
    # ---------- Technical Safety Controls ----------------------------------------
    'AI Safety Mechanism':         'mekanisme keamanan dan perlindungan konten berbahaya dalam sistem AI',
    'Content Moderation':          'moderasi konten berbahaya dalam layanan kecerdasan buatan',
    'Safety Testing / Red-teaming':'pengujian keamanan dan uji coba serangan pada sistem AI',
    'Incident Monitoring':         'pemantauan insiden keamanan pada sistem kecerdasan artifisial',
    # ---------- Liability & Governance -------------------------------------------
    'Liability Framework':         'kerangka tanggung jawab dan ganti rugi atas kerusakan akibat AI',
    'Regulatory Sandbox':          'ruang uji regulasi untuk inovasi kecerdasan artifisial',
    'Cross-border AI Governance':  'tata kelola AI lintas batas negara dan yurisdiksi',
    'Indonesian AI Governance':    'pengaturan tata kelola kecerdasan artifisial Indonesia',
    # ---------- Indonesian Digital Context ---------------------------------------
    'Hoaks / Misinformation':      'penyebaran berita hoaks dan disinformasi melalui AI',
    'Penipuan Online':             'kejahatan penipuan digital menggunakan kecerdasan buatan',
    'SARA Content':                'konten berbau SARA yang dihasilkan oleh sistem AI',
    'Data Privacy':                'privasi data pengguna dalam layanan kecerdasan artifisial',
    # ---------- Accountability ---------------------------------------------------
    'Developer Accountability':    'akuntabilitas dan kewajiban pengembang sistem kecerdasan buatan',
    'Algorithm Transparency':      'transparansi algoritma dan sistem kecerdasan artifisial',
    'Impact Assessment':           'penilaian dampak sistem kecerdasan artifisial terhadap masyarakat',
}

print(f"Defined {len(SAFETY_CONCEPTS)} safety/governance concepts for analysis.")
print("\nConcepts:")
for k in SAFETY_CONCEPTS:
    print(f"  • {k}")

In [ ]:
# ── Semantic Coverage Analysis ────────────────────────────────────────────────

def keyword_coverage_fallback(text: str, concepts: dict) -> dict:
    """Simple keyword-based coverage when sentence-transformers is unavailable."""
    text_lower = text.lower()
    return {
        concept: min(1.0, sum(kw.lower() in text_lower for kw in concept_text.split()) / 3)
        for concept, concept_text in concepts.items()
    }


def semantic_coverage(text: str, concepts: dict, model, chunk_size: int = 5000) -> dict:
    """
    Compute semantic similarity between document text (chunked) and each concept.
    Uses max-pooling over chunks: the most relevant passage drives the score.
    """
    # Split document into chunks to fit model context
    words = text.split()
    chunks = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    concept_labels = list(concepts.keys())
    concept_texts  = list(concepts.values())
    concept_embs   = model.encode(concept_texts, show_progress_bar=False)

    max_sims = np.zeros(len(concept_labels))
    for chunk in chunks:
        chunk_emb  = model.encode([chunk], show_progress_bar=False)
        sims       = cosine_similarity(chunk_emb, concept_embs)[0]
        max_sims   = np.maximum(max_sims, sims)

    return dict(zip(concept_labels, max_sims.tolist()))


# Load embedding model
coverage_results = {}
if SEMANTIC_AVAILABLE:
    print("Loading multilingual embedding model (paraphrase-multilingual-MiniLM-L12-v2)...")
    print("This may take ~1 min on first run (downloads ~450 MB).")
    embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    print("Model loaded.\n")

    for doc_id, text in cleaned_texts.items():
        short = DOCUMENTS[doc_id]['short']
        print(f"Analyzing {short}...", end=' ')
        scores = semantic_coverage(text, SAFETY_CONCEPTS, embed_model, chunk_size=3000)
        coverage_results[doc_id] = scores
        print(f"done. Top concept: {max(scores, key=scores.get)} ({max(scores.values()):.3f})")
else:
    print("Using keyword-based fallback for coverage analysis.")
    for doc_id, text in cleaned_texts.items():
        coverage_results[doc_id] = keyword_coverage_fallback(text, SAFETY_CONCEPTS)

# Build DataFrame
coverage_df = pd.DataFrame(coverage_results).T
coverage_df.index = [DOCUMENTS[d]['short'] for d in coverage_df.index]
print("\nCoverage DataFrame shape:", coverage_df.shape)
coverage_df.round(3)

In [ ]:
# ── Liability Actor Extraction (Rule-Based NLP) ───────────────────────────────

ACTOR_PATTERNS = {
    'Foundation Model Provider': [
        r'penyelenggara\s+sistem\s+elektronik\s+asing',
        r'platform\s+asing',
        r'layanan\s+digital\s+asing',
        r'penyedia\s+layanan\s+AI\b',
        r'perusahaan\s+teknologi\s+asing',
    ],
    'API Developer (Domestic)': [
        r'pengembang\s+aplikasi',
        r'penyelenggara\s+sistem\s+elektronik\b',
        r'penyedia\s+layanan\s+elektronik',
        r'pengelola\s+platform',
        r'pelaku\s+usaha\s+berbasis\s+teknologi',
    ],
    'End User': [
        r'pengguna\s+layanan',
        r'konsumen',
        r'masyarakat\s+pengguna',
        r'subjek\s+data',
    ],
    'Government / Regulator': [
        r'pemerintah\s+pusat',
        r'kementerian\s+yang\s+menyelenggarakan',
        r'badan\s+pengawas',
        r'lembaga\s+pengawas',
        r'komisi\s+pengawas',
        r'badan\s+siber',
        r'BSSN\b',
        r'Kominfo\b',
    ],
}

LIABILITY_KEYWORDS = [
    'bertanggung jawab', 'menanggung', 'wajib memberikan', 'kewajiban',
    'ganti rugi', 'sanksi', 'pidana', 'denda', 'larangan',
    'tanggung jawab hukum', 'tindak pidana',
]


def extract_actor_mentions(text: str) -> dict:
    """Count co-occurrence of liability keywords near each actor pattern."""
    text_lower = text.lower()
    result = {actor: {'mentions': 0, 'liability_context': 0} for actor in ACTOR_PATTERNS}

    for actor, patterns in ACTOR_PATTERNS.items():
        for pattern in patterns:
            matches = list(re.finditer(pattern, text_lower))
            result[actor]['mentions'] += len(matches)
            # Check 200-char window around each match for liability keywords
            for m in matches:
                window = text_lower[max(0, m.start()-200): m.end()+200]
                if any(kw in window for kw in LIABILITY_KEYWORDS):
                    result[actor]['liability_context'] += 1

    return result


actor_results = {}
for doc_id, text in cleaned_texts.items():
    actor_results[doc_id] = extract_actor_mentions(text)

# Summary table
rows = []
for doc_id, actors in actor_results.items():
    for actor, counts in actors.items():
        rows.append({
            'Document': DOCUMENTS[doc_id]['short'],
            'Actor': actor,
            'Mentions': counts['mentions'],
            'Liability_Context': counts['liability_context'],
        })

actor_df = pd.DataFrame(rows)
print("Actor Mention Summary:")
print(actor_df.pivot_table(index='Actor', columns='Document', values='Mentions', fill_value=0).to_string())

In [ ]:
# ── Regulatory Gap Matrix ─────────────────────────────────────────────────────

# Threshold: semantic similarity >= 0.35 = substantively covered
COVERAGE_THRESHOLD = 0.35

gap_matrix = coverage_df >= COVERAGE_THRESHOLD  # True = covered, False = gap

# Categorize gaps
gap_analysis = []
for concept in coverage_df.columns:
    scores = coverage_df[concept]
    covered_docs = [doc for doc in coverage_df.index if coverage_df.loc[doc, concept] >= COVERAGE_THRESHOLD]
    gap_analysis.append({
        'Concept': concept,
        'Max_Similarity': round(scores.max(), 3),
        'Avg_Similarity': round(scores.mean(), 3),
        'Covered_In': ', '.join(covered_docs) if covered_docs else 'NONE — Critical Gap',
        'Coverage_Count': len(covered_docs),
        'Is_Critical_Gap': len(covered_docs) == 0,
    })

gap_df = pd.DataFrame(gap_analysis).sort_values('Max_Similarity', ascending=False)

critical_gaps = gap_df[gap_df['Is_Critical_Gap'] == True]
print(f"Total concepts analyzed: {len(gap_df)}")
print(f"Concepts with NO coverage across all documents: {len(critical_gaps)}")
print("\n--- CRITICAL REGULATORY GAPS ---")
for _, row in critical_gaps.iterrows():
    print(f"  ⚠ {row['Concept']} (max similarity: {row['Max_Similarity']})")

# Export gap analysis
gap_path = PROCESSED_DIR / 'regulatory_gap_analysis.json'
gap_export = {
    'analysis_date': datetime.now().isoformat(),
    'method': 'semantic_embedding' if SEMANTIC_AVAILABLE else 'keyword_fallback',
    'model': 'paraphrase-multilingual-MiniLM-L12-v2' if SEMANTIC_AVAILABLE else 'keyword',
    'coverage_threshold': COVERAGE_THRESHOLD,
    'gap_analysis': gap_df.to_dict(orient='records'),
    'actor_analysis': actor_df.to_dict(orient='records'),
    'coverage_matrix': coverage_df.round(4).to_dict(),
}
gap_path.write_text(json.dumps(gap_export, indent=2, ensure_ascii=False), encoding='utf-8')
print(f"\nGap analysis exported: {gap_path}")

In [ ]:
# ── Visualizations ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Regulatory Corpus: AI Safety Coverage Analysis', fontsize=14, fontweight='bold', y=1.01)

# ── Plot 1: Full Coverage Heatmap ─────────────────────────────────────────────
ax1 = axes[0]
cmap = sns.diverging_palette(10, 130, as_cmap=True)  # red→green
sns.heatmap(
    coverage_df.T,
    ax=ax1,
    cmap=cmap,
    vmin=0, vmax=0.7,
    annot=True, fmt='.2f',
    linewidths=0.5,
    cbar_kws={'label': 'Semantic Similarity', 'shrink': 0.8},
)
ax1.set_title('Semantic Coverage by Document & Concept')
ax1.set_xlabel('Regulatory Document')
ax1.set_ylabel('AI Safety / Governance Concept')
ax1.tick_params(axis='y', labelsize=8)

# Add threshold line indicator via annotation
ax1.axhline(y=0, color='gray', linewidth=0.5)

# ── Plot 2: Actor Liability Context Bar Chart ─────────────────────────────────
ax2 = axes[1]
pivot_liability = actor_df.pivot_table(
    index='Actor', columns='Document', values='Liability_Context', fill_value=0
)
pivot_liability.plot(
    kind='bar',
    ax=ax2,
    colormap='Set2',
    edgecolor='black',
    linewidth=0.5,
    width=0.7,
)
ax2.set_title('Liability Mentions in Context by Actor & Document')
ax2.set_xlabel('Actor Category')
ax2.set_ylabel('Count of Liability-Context Mentions')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=30, ha='right')
ax2.legend(title='Document', fontsize=8)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig_path = DIAGRAMS_DIR / 'regulatory_coverage_analysis.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"\nFigure saved: {fig_path}")

In [ ]:
# ── API Governance Gap Focus ───────────────────────────────────────────────────
# Zoom into the API-specific concepts that are the core of this research

API_CONCEPTS = [
    'API Safety Obligation',
    'API Developer Liability',
    'Foundation Model Provider',
    'Third-party Deployment',
    'Safety Guardrail Stripping',
    'Safety Testing / Red-teaming',
    'Incident Monitoring',
    'Cross-border AI Governance',
    'Liability Framework',
]

api_coverage = coverage_df[API_CONCEPTS]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    api_coverage.T,
    ax=ax,
    cmap=sns.color_palette('RdYlGn', as_cmap=True),
    vmin=0, vmax=0.7,
    annot=True, fmt='.3f',
    linewidths=0.8,
    cbar_kws={'label': 'Semantic Similarity (0 = no coverage, 0.7+ = strong)', 'shrink': 0.6},
)
ax.set_title(
    'API Governance Gap Analysis: Indonesian Regulatory Coverage of API-Specific AI Safety Risks\n'
    f'(Threshold for substantive coverage = {COVERAGE_THRESHOLD}; red = gap)',
    fontsize=11
)
ax.set_xlabel('Regulatory Document')
ax.set_ylabel('API Safety / Governance Concept')
ax.tick_params(axis='y', labelsize=9)

# Add a horizontal dashed line as coverage threshold indicator
for i in range(len(API_CONCEPTS)):
    for j, doc in enumerate(api_coverage.index):
        val = api_coverage.loc[doc, API_CONCEPTS[i]]
        if val < COVERAGE_THRESHOLD:
            ax.add_patch(mpatches.Rectangle((j, i), 1, 1, fill=False, edgecolor='red', linewidth=2))

plt.tight_layout()
fig_path2 = DIAGRAMS_DIR / 'api_governance_gap_matrix.png'
plt.savefig(fig_path2, dpi=200, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path2}")
print("\nRed-bordered cells = CRITICAL REGULATORY GAPS for API AI Safety")

In [ ]:
# ── Final Narrative Report ────────────────────────────────────────────────────

covered_api = api_coverage.max() >= COVERAGE_THRESHOLD
gap_count   = (~covered_api).sum()

print("=" * 70)
print("REGULATORY CORPUS ANALYSIS — KEY FINDINGS")
print("=" * 70)
print()
print(f"Documents analyzed: {len(DOCUMENTS)}")
print(f"  • {', '.join(d['short'] for d in DOCUMENTS.values())}")
print()
print(f"Safety concepts evaluated: {len(SAFETY_CONCEPTS)}")
print(f"API-specific concepts evaluated: {len(API_CONCEPTS)}")
print(f"API concepts with NO substantive coverage: {gap_count} / {len(API_CONCEPTS)}")
print()
print("--- UNCOVERED API GOVERNANCE AREAS ---")
for concept in API_CONCEPTS:
    max_sim = api_coverage[concept].max()
    status = '✓ Partially Covered' if max_sim >= COVERAGE_THRESHOLD else '✗ CRITICAL GAP'
    best_doc = api_coverage[concept].idxmax()
    print(f"  {status:25s} {concept:35s} (best: {best_doc}, sim={max_sim:.3f})")
print()
print("--- ACTOR LIABILITY ASSESSMENT ---")
for doc in [d['short'] for d in DOCUMENTS.values()]:
    doc_actors = actor_df[actor_df['Document'] == doc]
    best_actor = doc_actors.loc[doc_actors['Liability_Context'].idxmax(), 'Actor']
    max_l = doc_actors['Liability_Context'].max()
    print(f"  {doc:20s}: Highest liability focus on '{best_actor}' ({max_l} in-context mentions)")
print()
print("CONCLUSION: Current Indonesian regulatory framework does not substantively")
print("address the specific risks of API-mediated AI deployment by third parties.")
print("The 'API Developer (Domestic)' actor category appears rarely in liability context,")
print("confirming the socio-technical blind spot identified in the research framework.")
print("=" * 70)